# Fine-tune YOLO11x cho camera giao thông trên Modal
Gắn Volume `traffic-training` tại `/mnt/traffic-training`, chọn GPU H100 trong sidebar. Mặc định notebook dùng `public_traffic_dataset_balanced.zip` đã được chuẩn bị từ dataset giao thông công khai; có thể đổi `DATASET_ZIP` nếu dùng dữ liệu khác. Chạy các cell từ trên xuống. `PILOT=True` chạy 3 epoch để kiểm tra; đổi sang `False` để train chính thức từ COCO pretrained. Mỗi lần thay dữ liệu hoặc cấu hình, đổi `RUN_NAME`.


In [ ]:
%uv pip install ultralytics pyyaml


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import torch
import yaml

VOLUME = Path('/mnt/traffic-training')
DATASET_ZIP = VOLUME / 'datasets/public_traffic_dataset_balanced.zip'
RUN_NAME = 'public_traffic_yolo11x_v1'
PILOT = True
EPOCHS = 3 if PILOT else 80
EXPECTED_NAMES = ['bicycle', 'car', 'motorcycle', 'bus', 'truck']
assert VOLUME.is_dir(), 'Gắn Volume traffic-training vào notebook'
if not DATASET_ZIP.is_file():
    expected_name = DATASET_ZIP.name
    matches = list(VOLUME.rglob(expected_name))
    matches += [p for p in (Path.cwd() / expected_name, Path('/root') / expected_name) if p.is_file()]
    if len(set(matches)) == 1:
        DATASET_ZIP = matches[0]
        print('Tìm thấy ZIP tại:', DATASET_ZIP)
    else:
        nearby = [str(p) for base in (VOLUME, Path.cwd(), Path('/root')) for p in base.glob('*.zip')]
        raise FileNotFoundError(f'Không thấy {expected_name} tại {DATASET_ZIP}. '                                f'ZIP tìm thấy: {nearby[:20]}. '                                'Mở Files, vào đúng thư mục trên Volume rồi upload hoặc sửa DATASET_ZIP.')
assert torch.cuda.is_available(), 'Chọn GPU H100 trong sidebar rồi khởi động lại kernel'
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in ('.', '..')
zip_sha = hashlib.sha256(DATASET_ZIP.read_bytes()).hexdigest()
dataset_root = VOLUME / 'datasets' / ('unpacked_' + zip_sha[:12])
data_yaml = dataset_root / 'traffic_dataset/data.yaml'
if not data_yaml.is_file():
    dataset_root.mkdir(parents=True, exist_ok=True)
    with ZipFile(DATASET_ZIP) as archive:
        for item in archive.infolist():
            target = (dataset_root / item.filename).resolve()
            if not target.is_relative_to(dataset_root.resolve()):
                raise ValueError(f'Unsafe ZIP member: {item.filename}')
        archive.extractall(dataset_root)
assert data_yaml.is_file(), 'ZIP cần chứa traffic_dataset/data.yaml'
data = yaml.safe_load(data_yaml.read_text(encoding='utf-8'))
assert data['names'] == EXPECTED_NAMES, f'Sai thứ tự lớp: {data["names"]}'
for split in ('train', 'val', 'test'):
    folder = data_yaml.parent / data[split]
    images = [p for p in folder.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    assert images, f'{split} không có ảnh: {folder}'
    print(split, len(images), 'ảnh')
data['path'] = str(data_yaml.parent.resolve())
runtime_yaml = data_yaml.parent / 'modal_data.yaml'
runtime_yaml.write_text(yaml.safe_dump(data, sort_keys=False), encoding='utf-8')
print('GPU:', torch.cuda.get_device_name(0))
print('ZIP SHA256:', zip_sha)


In [ ]:
from ultralytics import YOLO

project = VOLUME / 'runs'
run = RUN_NAME + ('_pilot' if PILOT else '_full')
run_dir = project / run
assert not run_dir.exists(), f'{run_dir} đã tồn tại; đổi RUN_NAME để tránh ghi đè'
model = YOLO('yolo11x.pt')  # COCO pretrained, cùng nguồn với weights/coco/yolo11x.pt
model.train(data=str(runtime_yaml), epochs=EPOCHS, imgsz=640, batch=4,
            device=0, workers=2, seed=42, optimizer='AdamW', lr0=0.001,
            project=str(project), name=run, exist_ok=False, plots=True)
best = run_dir / 'weights/best.pt'
assert best.is_file(), f'Thiếu checkpoint: {best}'
(run_dir / 'dataset_info.json').write_text(json.dumps({
    'zip_sha256': zip_sha, 'pretrained': 'yolo11x.pt', 'epochs': EPOCHS,
    'names': EXPECTED_NAMES, 'pilot': PILOT
}, indent=2), encoding='utf-8')
print('Checkpoint:', best)


In [ ]:
# Chỉ chạy sau khi đã chốt lần train chính thức. Test chỉ dùng để đánh giá cuối.
if not PILOT:
    metrics = YOLO(str(best)).val(data=str(runtime_yaml), split='test', imgsz=640,
                                   batch=4, device=0, project=str(project),
                                   name=run + '_test')
    print('Test mAP50-95:', metrics.box.map)
else:
    print('Pilot xong. Đặt PILOT=False rồi chạy lại từ cell cấu hình để train 80 epoch.')


Sau khi train, tải `best.pt` từ Volume về máy. Chạy thử bằng `python main.py --video <video.mp4> --vehicle-weights <best.pt>` (xem `python main.py --help` nếu tham số video thay đổi). Dừng kernel Modal khi hoàn tất.
